# 02 - Predictive Modeling
EDA found no significant linear or categorical association between network variables and Efficiency_Status. This notebook builds a classifier using ALL variables to quantify, numerically, how much each variable (including network variables) actually matters for predicting efficiency — a more sensitive test than pairwise correlation, since models can pick up combined/interaction effects that single-variable tests miss.

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/Thales_Group_Manufacturing.csv')
df.head()

,Date,Timestamp,Machine_ID,Operation_Mode,Temperature_C,Vibration_Hz,Power_Consumption_kW,Network_Latency_ms,Packet_Loss_%,Quality_Control_Defect_Rate_%,Production_Speed_units_per_hr,Predictive_Maintenance_Score,Error_Rate_%,Efficiency_Status
0,01-01-2025,00:00:00,39,Idle,74.138,3.501,8.612,10.651,0.208,7.751,477.657,0.345,14.965,Low
1,01-01-2025,00:01:00,29,Active,84.265,3.356,2.269,29.112,2.228,4.989,398.175,0.770,7.678,Low
2,01-01-2025,00:02:00,15,Active,44.280,2.080,6.144,18.357,1.639,0.457,108.075,0.987,8.198,Low
3,01-01-2025,00:03:00,43,Active,40.569,0.298,4.068,29.154,1.161,4.583,329.579,0.983,2.741,Medium
4,01-01-2025,00:04:00,8,Idle,75.064,0.346,6.226,34.029,4.797,2.288,159.114,0.573,12.101,Low


In [11]:
features = ['Temperature_C', 'Vibration_Hz', 'Power_Consumption_kW',
            'Network_Latency_ms', 'Packet_Loss_%', 
            'Predictive_Maintenance_Score']

X = df[features]
y = df['Efficiency_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(80000, 6) (20000, 6)


In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        High       0.00      0.00      0.00       597
         Low       0.78      0.94      0.85     15565
      Medium       0.20      0.06      0.09      3838

    accuracy                           0.74     20000
   macro avg       0.33      0.33      0.32     20000
weighted avg       0.64      0.74      0.68     20000



In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

Temperature_C                   0.169005
Network_Latency_ms              0.167590
Power_Consumption_kW            0.167501
Packet_Loss_%                   0.167420
Vibration_Hz                    0.167178
Predictive_Maintenance_Score    0.161306
dtype: float64


**Data leakage detected:** feature importances showed Error_Rate_% (60%) and Production_Speed_units_per_hr (39%) accounting for 99.4% of the model's predictions — both are output/quality variables likely used to construct Efficiency_Status directly, not independent predictors. Rebuilding the feature set below to exclude Error_Rate_%, Production_Speed_units_per_hr, and Quality_Control_Defect_Rate_% (same category of variable).

**Result (leak-free model):** After removing outcome variables, the model performs poorly on the classes that matter — 0.00 recall on High, 0.06 recall on Medium, with only Low (the majority class at 78%) predicted well (0.94 recall). Overall accuracy of 0.74 is misleading given this imbalance. This confirms, consistent with the correlation and chi-square findings, that genuine input variables (mechanical and network) do not meaningfully predict Efficiency_Status in this dataset — network performance shows no detectable independent effect on manufacturing efficiency.